In [1]:
from importlib import reload
import torch
import numpy as np
import time
torch.set_default_dtype(torch.float64)
np.random.seed(2)
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(device)
import numpy as np
import sys
sys.path.append(r"../main_code/3d")
import generate_data3,grid_cell,visual,cal_S_grad,org_m,mea_3D,net_3D_fix,S_valgrad,error_general_3D
CUDA_LAUNCH_BLOCKING=1

def f_y(x, x_min,x_max,y_min,y_max,z_min,z_max,center,r):
    r_y = np.linalg.norm(x - center, axis=-1) 
    f_y_values = np.where(r_y < r, r - r_y, 0.0)
    return f_y_values

def ana_S(x,x_min,x_max,y_min,y_max,z_min,z_max,a,b,r):
    return f_y(x,x_min,x_max,y_min,y_max,z_min,z_max,a,r)-f_y(x,x_min,x_max,y_min,y_max,z_min,z_max,b,r)


cuda:1


## 81 5%

In [2]:
import copy
kk=[1,5,9,13,17,21,25,29,33,37,41,45,49,53,57,61,65,69,73,77,81]
tau_x_min,tau_x_max,tau_y_min,tau_y_max,tau_z_min,tau_z_max=0,1,0,1,0,1
b_x_min,b_x_max,b_y_min,b_y_max,b_z_min,b_z_max=-0.5,1.5,-0.5,1.5,-0.5,1.5 #边界gamma处
num_batches_gauss=1
b_n=10
points_b=mea_3D.generate_cube_surface(b_x_min,b_x_max,b_y_min,b_y_max,b_z_min,b_z_max,b_n) #在边界Γ处收集数据
x_left,x_right,y_below,y_upper=0.29,0.49,0.3,0.7
r_true=0.2
center_1=np.array([0.3,0.5,0.3])
center_2=np.array([0.5,0.5,0.8])
center_true=np.array([[0.3,0.5,0.3],[0.5,0.5,0.8]])
number_gene=50
eps=0.05
batch_number_rec_mea,mea=2,1
cupy_device=1
num_batches_gauss=4
num_batches_appr=10
num_batches_mea=1
S_int_true1, F_mea_b1, F_mea_db_x11, F_mea_db_x21,F_mea_db_x31 = generate_data3.boundary_data_circle(kk, number_gene, points_b, f_y,tau_x_min,tau_x_max,tau_y_min,tau_y_max,tau_z_min,tau_z_max,center_1, r_true, eps,mea,num_batches_mea,device,cupy_device)
S_int_true2, F_mea_b2, F_mea_db_x12, F_mea_db_x22,F_mea_db_x32 =generate_data3.boundary_data_circle(kk, number_gene, points_b, f_y,tau_x_min,tau_x_max,tau_y_min,tau_y_max,tau_z_min,tau_z_max,center_2, r_true, eps,mea,num_batches_mea,device,cupy_device)
F_mea_b0 = F_mea_b1 - F_mea_b2
F_mea_db_x10 = F_mea_db_x11 - F_mea_db_x12
F_mea_db_x20= F_mea_db_x21 -F_mea_db_x22
F_mea_db_x30= F_mea_db_x31 -F_mea_db_x32
F=org_m.F(F_mea_b0,F_mea_db_x10,F_mea_db_x20,F_mea_db_x30)

In [6]:
np.save("./noise=5%/F_81_5%.npy",F)